# IF3270 Pembelajaran Mesin — Tugas Besar 1
## FFNN From Scratch: Global Student Placement & Salary


## 1. Import Required Libraries

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from activations import get_activation
from losses import get_loss
from layer import DenseLayer
from ffnn import FFNN
from utils import get_weight_stats, get_gradient_stats, loss_curve, weight_histograms, gradient_histograms

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 4)


## 2. Load and Explore the Dataset

In [ ]:
DATA_PATH = '../data/student_placement_salary.csv'

df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
df.head()


In [ ]:
print(df.dtypes)
print("\nMissing values:\n", df.isnull().sum())
df.describe()

## 3. Data Preprocessing

In [ ]:
TARGET_COL  = 'PlacementStatus'   

# 1. Drop rows with missing target
df = df.dropna(subset=[TARGET_COL])

# 2. Encode categorical features
cat_cols = df.select_dtypes(include='object').columns.tolist()
cat_cols = [c for c in cat_cols if c != TARGET_COL]

le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

# 3. Encode target
y_raw = le.fit_transform(df[TARGET_COL].astype(str))
n_classes = len(np.unique(y_raw))

if n_classes == 2:
    # Binary classification → sigmoid + BCE
    y = y_raw.reshape(-1, 1).astype(float)
    TASK = 'binary'
else:
    # Multi-class → softmax + CCE (one-hot)
    y = np.eye(n_classes)[y_raw]
    TASK = 'multiclass'

print(f"Task: {TASK}, classes: {n_classes}, y shape: {y.shape}")

# 4. Feature matrix
X_df = df.drop(columns=[TARGET_COL])
X_df = X_df.select_dtypes(include=[np.number]).fillna(0)

scaler = StandardScaler()
X = scaler.fit_transform(X_df.values).astype(float)
print(f"X shape: {X.shape}")

# 5. Train / validation / test split  (60 / 20 / 20)
X_tmp,  X_test,  y_tmp,  y_test  = train_test_split(X, y, test_size=0.20, random_state=42)
X_train, X_val,  y_train, y_val  = train_test_split(X_tmp, y_tmp, test_size=0.25, random_state=42)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")


## 4. Build the Model

In [ ]:
n_features = X_train.shape[1]
n_out      = y_train.shape[1]

INIT = {'init_method': 'random_normal', 'init_params': {'mean': 0, 'var': 0.01, 'seed': 42}}

layer_configs = [
    {'n_in': n_features, 'n_out': 64, 'activation': 'relu',    **INIT},
    {'n_in': 64,         'n_out': 32, 'activation': 'relu',    **INIT},
    {'n_in': 32,         'n_out': n_out,
     'activation': 'softmax' if TASK == 'multiclass' else 'sigmoid', **INIT},
]

model = FFNN(layer_configs)

LOSS = 'cce' if TASK == 'multiclass' else 'bce'
model.compile(loss=LOSS, lr=0.01, regularization='l2', lambda_=1e-4)

model.summary()


## 5. Train the Model

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs     = 200,
    batch_size = 32,
    X_val      = X_val,
    y_val      = y_val,
    verbose    = 10,
)

## 6. Evaluate the Model

In [ ]:
y_prob = model.predict(X_test)

if TASK == 'binary':
    y_pred_cls = (y_prob >= 0.5).astype(int).ravel()
    y_true_cls = y_test.astype(int).ravel()
else:
    y_pred_cls = np.argmax(y_prob, axis=1)
    y_true_cls = np.argmax(y_test, axis=1)

accuracy = np.mean(y_pred_cls == y_true_cls)
test_loss = model.loss_fn.forward(y_prob, y_test)

print(f"Test loss    : {test_loss:.6f}")
print(f"Test accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

# Confusion matrix (manual)
from collections import Counter
cm = np.zeros((n_classes, n_classes), dtype=int)
for t, p in zip(y_true_cls, y_pred_cls):
    cm[t, p] += 1
print("\nConfusion matrix (rows=true, cols=pred):\n", cm)


## 7. Visualize Results

In [ ]:
train_losses, val_losses = loss_curve(history)

plt.figure(figsize=(10, 4))
plt.plot(train_losses, label='Train loss')
if val_losses is not None:
    plt.plot(val_losses, label='Validation loss', linestyle='--')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training / Validation Loss Curve')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
hists = weight_histograms(model)
n_layers = len(hists)

fig, axes = plt.subplots(1, n_layers, figsize=(5 * n_layers, 4))
if n_layers == 1: axes = [axes]

for ax, h in zip(axes, hists):
    widths = np.diff(h['bins'])
    ax.bar(h['bins'][:-1], h['counts'], width=widths, align='edge', alpha=0.7)
    ax.set_title(h['label'])
    ax.set_xlabel('Weight value')
    ax.set_ylabel('Count')

fig.suptitle('Weight Distributions After Training', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
_ = model.forward(X_train[:32])
model.backward(_, y_train[:32])

ghists = gradient_histograms(model)

fig, axes = plt.subplots(1, n_layers, figsize=(5 * n_layers, 4))
if n_layers == 1: axes = [axes]

for ax, h in zip(axes, ghists):
    widths = np.diff(h['bins'])
    ax.bar(h['bins'][:-1], h['counts'], width=widths, align='edge',
           alpha=0.7, color='tomato')
    ax.set_title(h['label'])
    ax.set_xlabel('Gradient value')
    ax.set_ylabel('Count')

fig.suptitle('Gradient Distributions (Last Batch)', fontsize=13)
plt.tight_layout()
plt.show()


## 8. Save & Load Model

In [ ]:
model.save('../models/ffnn_model.npz')

model2    = FFNN.load('../models/ffnn_model.npz')
y_prob2   = model2.predict(X_test)
max_diff  = np.max(np.abs(y_prob - y_prob2))
print(f"Max prediction difference after save/load: {max_diff:.2e}")
assert max_diff < 1e-10, "Predictions differ — save/load error!"
print("Save/load OK.")